# Lab-2a: Fine-Tuning Data Preparation (Text Summarization)

**Persona:** Data Engineer &nbsp;|&nbsp; **Lab:** 2 - Data Prep

## Overview

This notebook prepares the **instruction-tuning dataset** consumed by the fine-tuning lab (`lab3-model-build/lab-3b-fine-tuning.ipynb`). Data prep is a standalone, reusable step that publishes a ready-to-train dataset to Amazon S3 rather than preprocessing inline in the training notebook.

It supports **two interchangeable data sources**, selected by a single flag in Section 1:

| `DATA_SOURCE` | Corpus | Needs the internet? | Teaches |
|---|---|---|---|
| `"dolly"` | Databricks Dolly 15k, filtered to the summarization category | Yes — Hugging Face Hub | Acquiring, filtering, and validating a third-party corpus |
| `"synthetic"` | Generated in-notebook from `synthetic_corpus.py` | No | Designing a corpus when no suitable dataset exists |

Both paths produce the **same output contract** at the **same S3 location**, so Lab 3B runs unchanged either way.

### Why offer both

Downloading a public corpus is the common case, and it is what you will usually do. But it fails in ways that have nothing to do with your skill:

- The Hub is unreachable from a locked-down VPC, blocked by a proxy, or briefly down
- Dataset licensing and provenance need review before customer use
- Public corpora contain PII and content you did not vet
- Your actual domain has no public dataset at all
- Dolly is ~15k records, of which only ~1.1k are usable for summarization

The synthetic path removes the dependency entirely. It is also the honest answer to "what do I do when the dataset does not exist", which is the situation most real fine-tuning projects start from.

### Structure

Only **Section 2** differs by source. Everything after it is shared:

```
Section 1  Setup and source selection
Section 2  Acquire the corpus        <- the only branch
              Path A: load Dolly from the Hub
              Path B: generate offline
Section 3  Quality gates             <- shared
Section 4  Profile the dataset       <- shared
Section 5  Split and leakage check   <- shared
Section 6  Write the package         <- shared
Section 7  Upload to S3              <- shared
```

That shape is deliberate. Validation, profiling, splitting, and leakage checking are **source-independent** — the fact that a corpus came from a reputable public dataset is not evidence that it is fit for your training job.

### Output contract (consumed by Lab 3B)

All files are uploaded to `s3://<default-bucket>/<ProfileName>/dolly_dataset/`:

- `train.jsonl` — training examples, `{instruction, context, response}`
- `test.jsonl` — held-out evaluation examples
- `template.json` — prompt template for JumpStart instruction fine-tuning

### Data format for instruction tuning

```json
{
  "instruction": "Summarize the following text",
  "context": "[Long text to summarize]",
  "response": "[Expected summary]"
}
```

---
## Section 1: Setup and Source Selection

Dependencies come from the workshop's root `requirements.txt` (installed once via `setup.ipynb`), so there is no `pip install` cell here.

In [40]:
import os, sys
# Find the repo root (holds workshop_env.py) and import the shared loader.
from pathlib import Path
for _c in [os.getcwd(), *[str(p) for p in Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_c, 'workshop_env.py')):
        sys.path.insert(0, _c); break
from workshop_env import load_workshop_env
env = load_workshop_env()
SAGEMAKER_EXEC_ROLE = os.environ.get("SAGEMAKER_EXEC_ROLE", get_execution_role)
print(SAGEMAKER_EXEC_ROLE)

arn:aws:iam::716664005094:role/smai-genai-ml-std-product-role-


In [41]:
import hashlib
import json
import os
import random
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

from sagemaker.core.helper.session_helper import Session, get_execution_role

# The synthetic generator lives beside this notebook so the notebook itself stays
# readable. Adding the notebook directory to sys.path makes the import work
# regardless of the kernel's working directory.
notebook_dir = Path.cwd()
if not (notebook_dir / "synthetic_corpus.py").exists() and (notebook_dir / "lab2-data-prep").exists():
    notebook_dir = notebook_dir / "lab2-data-prep"
sys.path.insert(0, str(notebook_dir))

print("✓ Imports complete")

✓ Imports complete


### Choose your data source

Set `DATA_SOURCE` to either `"dolly"` or `"synthetic"`. This is the **only** cell you change to switch paths — nothing downstream needs editing.

Guidance:

- **`"dolly"`** — the default. Use it when the Studio environment can reach the Hugging Face Hub and you want the lab to exercise a real, human-written corpus.
- **`"synthetic"`** — use it when the Hub is blocked, when a download fails mid-workshop, or when the teaching goal is dataset *design* rather than dataset *acquisition*.

In [42]:
# ---------------------------------------------------------------------------
# THE ONE SWITCH: "dolly" or "synthetic"
# ---------------------------------------------------------------------------
DATA_SOURCE = "synthetic"
# ---------------------------------------------------------------------------

VALID_SOURCES = ("dolly", "synthetic")
if DATA_SOURCE not in VALID_SOURCES:
    raise ValueError(f"DATA_SOURCE must be one of {VALID_SOURCES}, got {DATA_SOURCE!r}")

USING_DOLLY = DATA_SOURCE == "dolly"
USING_SYNTHETIC = DATA_SOURCE == "synthetic"

print(f"Data source: {DATA_SOURCE}")
print("  Dolly 15k from the Hugging Face Hub (requires internet access)" if USING_DOLLY
      else "  Generated offline by synthetic_corpus.py (no network required)")

Data source: synthetic
  Generated offline by synthetic_corpus.py (no network required)


In [43]:
# SageMaker session. Used to resolve the bucket and upload the finished package.
sess = Session()
region = sess.boto_region_name

# Account/region default SageMaker bucket: sagemaker-<region>-<account-id>.
# Lab 2B writes to this same bucket under a different top-level prefix
# (bank-marketing-lab/...), so the two labs never collide.
# NOTE: Lab 3B must resolve to this SAME bucket so the fine-tuning job finds the data.
bucket = sess.default_bucket()

# ---------------------------------------------------------------------------
# Shared values come from the repo-root .env that lab0-setup/setup.ipynb wrote.
# ---------------------------------------------------------------------------
# Read straight from the .env FILE rather than os.environ, on purpose: Studio
# pre-seeds some of these -- notably MLFLOW_TRACKING_URI, which it points at the
# domain's DEFAULT app -- and the shared loader uses override=False, so an
# ambient value silently wins over the project's own. Lab 3B resolves its MLflow
# app the same way, for the same reason.
for _candidate in [os.getcwd(), *[str(p) for p in Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_candidate, "workshop_env.py")):
        sys.path.insert(0, _candidate)
        break


def env_from_file(key, default=""):
    """Return one key's value from the repo-root .env, or `default`."""
    try:
        from workshop_env import find_env

        env_path = find_env()
        if env_path:
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if line.startswith(f"{key}="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    except Exception as exc:
        print(f"  (could not read .env: {type(exc).__name__}: {exc})")
    return default


# Execution role: discovered once by lab0-setup from the deployed domain, so
# every lab agrees on it. get_execution_role() is kept only as a fallback --
# it needs a SageMaker execution context and raises on a local kernel, which is
# precisely where reading .env helps.
role = env_from_file("SAGEMAKER_EXEC_ROLE")
role_source = ".env (SAGEMAKER_EXEC_ROLE)"
if not role:
    try:
        role = get_execution_role()
        role_source = "get_execution_role()"
    except Exception as exc:
        role, role_source = "", f"unresolved ({type(exc).__name__})"
        print("⚠ No SAGEMAKER_EXEC_ROLE in .env and get_execution_role() failed.")
        print("  Run lab0-setup/setup.ipynb to populate .env. This lab only")
        print("  reports the role, so data prep still completes without it.")

if role and not role.startswith("arn:aws:iam::"):
    print(f"⚠ SAGEMAKER_EXEC_ROLE is not a role ARN: {role!r}")

print(f"Amazon SageMaker role: {role or '(unresolved)'}")
print(f"  resolved from:       {role_source}")
print(f"Amazon S3 bucket:      {bucket}")
print(f"AWS Region:            {region}")

Amazon SageMaker role: arn:aws:iam::716664005094:role/smai-genai-ml-std-product-role-
  resolved from:       .env (SAGEMAKER_EXEC_ROLE)
Amazon S3 bucket:      sagemaker-us-east-1-716664005094
AWS Region:            us-east-1


In [44]:
# Studio user profile name: namespaces this user's S3 prefix.
# Resolution order matters. The .env value comes FIRST because lab0-setup
# discovers it once from the domain, which makes Studio and local kernels agree.
# The old code went straight to the metadata file and otherwise defaulted to
# 'userA', so the same notebook wrote s3://.../Usera/ in Studio and
# s3://.../UserA/ locally -- two datasets, silently.
profile_name = env_from_file("SAGEMAKER_USER_PROFILE")
_profile_source = ".env (SAGEMAKER_USER_PROFILE)"
if not profile_name:
    metadata_path = Path("/opt/ml/metadata/resource-metadata.json")
    if metadata_path.exists():
        profile_name = json.loads(metadata_path.read_text())["UserProfileName"]
        _profile_source = "Studio resource-metadata.json"
    else:
        profile_name = os.environ.get("SM_USER_PROFILE_NAME", "")
        _profile_source = "SM_USER_PROFILE_NAME env var"
if not profile_name:
    raise RuntimeError(
        "Could not resolve the Studio user profile name, which the S3 dataset "
        "prefix is namespaced by. Run lab0-setup/setup.ipynb to populate "
        "SAGEMAKER_USER_PROFILE in .env, or set SM_USER_PROFILE_NAME. Guessing "
        "it would silently read or write the wrong user's prefix."
    )
# Capitalize the first letter to match the existing S3 layout.
profile_name = profile_name[0].upper() + profile_name[1:]

# Lab 3B resolves its data location to "<profile>/dolly_dataset". Both sources
# publish to that prefix so Lab 3B needs no edit either way — the prefix names
# the lab's slot in S3, not the corpus that filled it.
DATASET_PREFIX = "dolly_dataset"
data_location = f"s3://{bucket}/{profile_name}/{DATASET_PREFIX}"

print(f"Profile name:   {profile_name}")
print(f"Data location:  {data_location}")

Profile name:   Usera
Data location:  s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset


In [45]:
# Shared knobs. SEED makes the synthetic corpus and both splits reproducible —
# same seed in, byte-identical dataset out.
SEED = 42
TEST_SIZE = 0.3              # 70/30 train/test, as in the original Lab 2A
EXAMPLES_PER_DOMAIN = 200    # synthetic only: 6 domains -> 1,200 records,
                             # comparable to Dolly's ~1,160 summarization records

TRAINING_FIELDS = ("instruction", "context", "response")

print(f"Seed: {SEED}  |  Test split: {TEST_SIZE:.0%}")

Seed: 42  |  Test split: 30%


---
## Section 2: Acquire the Corpus

This is the only section that branches. Both paths define a loader function; the dispatch cell at the end of the section calls the one you selected and produces `raw_records` — a list of dicts with `instruction`, `context`, `response`, and a `domain` label used later for stratification and profiling.

Defining both loaders is harmless: a function definition has no side effects, and keeping them side by side makes the contract they share explicit.

### Path A — Databricks Dolly 15k

Dolly contains ~15,000 human-written instruction-following examples across categories including summarization, question answering, information extraction, creative writing, and classification. We keep only the **summarization** category to build a domain-specific model.

Records arrive with exactly the fields we need (`instruction`, `context`, `response`) once `category` is dropped. They are **human-written**, which matters for how we validate them in Section 3: real summaries paraphrase, infer, and occasionally wander from the source text.

In [46]:
def load_dolly_corpus():
    """Load Dolly 15k from the Hugging Face Hub and keep the summarization split.

    Requires outbound internet access from the kernel. If this raises a network
    or resolution error, switch DATA_SOURCE to "synthetic" and re-run from
    Section 1 — no other change is needed.
    """
    from datasets import load_dataset

    dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
    print(f"  Loaded {dolly.num_rows} Dolly records across {len(set(dolly['category']))} categories")

    summarization = dolly.filter(lambda example: example["category"] == "summarization")
    summarization = summarization.remove_columns("category")
    print(f"  Kept {summarization.num_rows} summarization records")

    # Normalize to plain dicts so both paths hand the same structure downstream.
    # Dolly has no sub-category, so every record shares one stratum.
    return [
        {
            "instruction": record["instruction"],
            "context": record["context"],
            "response": record["response"],
            "domain": "summarization",
        }
        for record in summarization
    ]


print("✓ Path A loader defined (Dolly / Hugging Face Hub)")

✓ Path A loader defined (Dolly / Hugging Face Hub)


### Path B — Offline synthetic generation

The generator lives in `synthetic_corpus.py` beside this notebook. It is template-grounded: each record begins as a structured **fact dictionary**, and the prose is a *rendering* of that dictionary.

Per domain, the module supplies:

- **`core`** sentence groups — one phrasing is chosen from each group and always included. Together the core sentences contain every fact a summary can cite.
- **`filler`** — plausible sentences carrying no summary-bearing fact. These are the distractors that make the task non-trivial: the model has to learn *what to leave out*.
- **`summary`** templates — these cite only core facts.

Which gives the invariant that makes the corpus trainable:

```
summary facts  ⊆  core sentence facts  ⊆  context
```

A target summary therefore **cannot** reference a fact its context does not state. Groundedness is a property of the construction, not something to hope for and then measure. Six domains ship by default: product launches, quarterly earnings, research studies, city infrastructure, sports results, and weather events.

Open `synthetic_corpus.py` to read or extend the templates. Adding a domain means adding one dict — the notebook picks it up with no changes.

In [47]:
import synthetic_corpus


def generate_synthetic_corpus():
    """Generate the offline corpus. No network, no API key, no model endpoint."""
    records = synthetic_corpus.generate(
        examples_per_domain=EXAMPLES_PER_DOMAIN,
        seed=SEED,
    )
    print(f"  Generated {len(records)} records across {len(synthetic_corpus.DOMAINS)} domains")
    return records


print(f"✓ Path B loader defined (synthetic_corpus.py, {len(synthetic_corpus.DOMAINS)} domains)")
for name in synthetic_corpus.DOMAINS:
    core, summaries = synthetic_corpus.phrasing_combinations(name)
    print(f"    {name:22s} {core:3d} core phrasing combinations x {summaries} summary variants")

✓ Path B loader defined (synthetic_corpus.py, 6 domains)
    product_launch          27 core phrasing combinations x 2 summary variants
    quarterly_earnings      27 core phrasing combinations x 2 summary variants
    research_study          27 core phrasing combinations x 2 summary variants
    city_infrastructure     27 core phrasing combinations x 2 summary variants
    sports_event             9 core phrasing combinations x 2 summary variants
    weather_event           27 core phrasing combinations x 2 summary variants


### Dispatch

One cell, one branch. Everything below this point is identical for both sources.

In [48]:
print(f"Acquiring corpus via DATA_SOURCE = {DATA_SOURCE!r}\n")

if USING_DOLLY:
    raw_records = load_dolly_corpus()
else:
    raw_records = generate_synthetic_corpus()

print(f"\n✓ {len(raw_records)} raw records ready for validation")
print(f"  Domains: {dict(Counter(r['domain'] for r in raw_records))}")

Acquiring corpus via DATA_SOURCE = 'synthetic'

  Generated 1200 records across 6 domains

✓ 1200 raw records ready for validation
  Domains: {'city_infrastructure': 200, 'quarterly_earnings': 200, 'weather_event': 200, 'sports_event': 200, 'research_study': 200, 'product_launch': 200}


In [49]:
# Inspect one record end to end.
example = raw_records[0]
print(f"DOMAIN:      {example['domain']}\n")
print(f"INSTRUCTION: {example['instruction']}\n")
print(f"CONTEXT:     {example['context'][:600]}{'...' if len(example['context']) > 600 else ''}\n")
print(f"RESPONSE:    {example['response']}")

DOMAIN:      city_infrastructure

INSTRUCTION: Summarize the following text.

CONTEXT:     The city of Fukuoka approved a bridge retrofit with a budget of 1650 million dollars. A public comment period drew a few hundred written responses, most of them supportive. An earlier version of the plan was shelved during the last budget cycle. Preparation work starts in November, according to transport director Tomas Egeland. The build is planned over 2 years, with projected daily use of 272000 people. Utility relocation surveys were completed over the winter.

RESPONSE:    A bridge retrofit budgeted at 1650 million dollars was approved in Fukuoka; it runs 2 years, is forecast to serve 272000 people a day, and begins site prep in November under transport director Tomas Egeland.


---
## Section 3: Quality Gates

The original version of this lab uploaded whatever the Hub returned. The README claimed a "validate — remove empty or too-short records" step that the notebook never actually performed. Dolly does contain summarization records with an empty `context`, and training on those teaches a model to produce a summary from nothing.

Six checks, each mapping to a real failure mode:

| Gate | Catches | Why it matters for fine-tuning |
|---|---|---|
| **Schema** | missing or empty `instruction` / `context` / `response` | JumpStart instruction tuning requires all three; an empty target teaches the model to emit nothing |
| **Minimum length** | trivially short context or summary | Nothing to summarize, or a target too short to carry the task |
| **Groundedness** | a number or proper noun in the summary that is absent from the context | This is a **hallucination label in your training data**. Train on it and you are teaching the model to invent facts |
| **Compression ratio** | summary nearly as long as, or longer than, the context | Teaches copying instead of summarizing |
| **Exact duplicates** | identical normalized contexts | Wasted steps and inflated apparent dataset size |
| **Near duplicates** | high token-set overlap within a domain | Over-represents one phrasing, which shows up as memorization |

### Enforcement depends on who wrote the data

The same check means different things depending on the corpus, so each gate is either **enforced** (drop the record) or **advisory** (count and report it):

| Gate | `synthetic` | `dolly` |
|---|---|---|
| Schema, minimum length | enforce | enforce |
| Exact and near duplicates | enforce | enforce |
| Groundedness | **enforce** | **advisory** |
| Compression ratio | **enforce** | **advisory** |

The reasoning matters more than the table. For the synthetic corpus **you own the generator**, so a groundedness failure is a template bug — enforce it, see zero drops, and treat any drop as a defect to fix at the source. For Dolly, the summaries are **human-written and abstractive**: people legitimately paraphrase, infer, and use words absent from the source. Enforcing a strict groundedness rule there would discard a large fraction of a perfectly good corpus and bias whatever survived toward extractive summaries. So you measure it, look at the number, and decide — rather than letting a heuristic silently reshape someone else's dataset.

That asymmetry is the main thing worth taking away from this section. A validator is not neutral; it encodes an assumption about how the data was produced.

### Two implementation details that are easy to get wrong

Both of these were real bugs caught while building this lab:

1. **The number regex must not absorb a trailing period.** A naive `\d[\d,\.]*` turns `"6050."` at the end of a sentence into the token `6050.`, which then fails to match `6050` in the context. That one character produced **310 false hallucination reports** on an early run and silently gutted one domain from 220 records to 64.
2. **Sentence-initial capitalized words are not entities.** `"Headcount stood at 6050."` capitalizes `Headcount` by grammar, not because it names something. Treating it as a proper noun flags ordinary paraphrase as hallucination. Hyphenated names such as `Mei-Lin` need the same care, since word tokenizers split them.

A validator that is too strict is not "safe". It quietly throws away good data and skews your distribution, and unlike a crash, nothing tells you it happened.

In [50]:
WORD = re.compile(r"[A-Za-z0-9']+")
# Trailing sentence periods must NOT join the number: "6050." has to read as 6050.
NUMBER = re.compile(r"\d+(?:[.,]\d+)*")
CAPITALIZED = re.compile(r"\b[A-Z][a-zA-Z\-]+\b")
SENTENCE = re.compile(r"(?<=[.!?])\s+")

MIN_CONTEXT_WORDS = 25
MIN_RESPONSE_WORDS = 8
MIN_COMPRESSION = 0.05
MAX_COMPRESSION = 0.60
NEAR_DUP_THRESHOLD = 0.92


def tokens(text):
    return WORD.findall(text.lower())


def check_schema(record):
    for field in TRAINING_FIELDS:
        value = record.get(field)
        if not isinstance(value, str) or not value.strip():
            return False, f"missing or empty field: {field}"
    if len(tokens(record["context"])) < MIN_CONTEXT_WORDS:
        return False, f"context shorter than {MIN_CONTEXT_WORDS} words"
    if len(tokens(record["response"])) < MIN_RESPONSE_WORDS:
        return False, f"response shorter than {MIN_RESPONSE_WORDS} words"
    return True, ""


def proper_nouns(text):
    """Capitalized tokens that are not sentence-initial.

    Sentence-initial words are capitalized by grammar, not because they name an
    entity. Including them would flag legitimate paraphrase such as
    "Headcount stood at 6050." as a hallucinated entity.
    """
    found = []
    for sentence in SENTENCE.split(text):
        words = sentence.split()
        for position, word in enumerate(words):
            if position == 0:
                continue
            match = CAPITALIZED.fullmatch(word.strip(".,;:()\"'"))
            if match:
                found.append(match.group(0))
    return found


def check_grounded(record):
    """Every number and proper noun in the summary must be present in the context."""
    context_numbers = set(NUMBER.findall(record["context"]))
    context_words = set(tokens(record["context"]))

    for number in NUMBER.findall(record["response"]):
        if number not in context_numbers:
            return False, f"ungrounded number: {number}"

    for entity in proper_nouns(record["response"]):
        # Hyphenated names tokenize into parts, so verify each part.
        if not all(part in context_words for part in tokens(entity)):
            return False, f"ungrounded entity: {entity}"

    return True, ""


def compression_ratio(record):
    return len(tokens(record["response"])) / max(len(tokens(record["context"])), 1)


def context_fingerprint(record):
    return hashlib.sha1(" ".join(tokens(record["context"])).encode()).hexdigest()


def jaccard(left, right):
    return len(left & right) / len(left | right) if (left or right) else 0.0


print("✓ Validators defined")

✓ Validators defined


In [51]:
# Which gates drop a record, and which only report. See the rationale above:
# you own the synthetic generator, so a failure there is a bug to fix; Dolly's
# human summaries are abstractive, so the same signal is information, not a defect.
ENFORCE, ADVISE = "enforce", "advise"

GATE_POLICIES = {
    "synthetic": {"schema": ENFORCE, "grounded": ENFORCE, "compression": ENFORCE, "duplicates": ENFORCE},
    "dolly":     {"schema": ENFORCE, "grounded": ADVISE,  "compression": ADVISE,  "duplicates": ENFORCE},
}


def run_quality_gates(records, policy):
    """Apply the gates under `policy`.

    Returns (kept, dropped, advisories, counts) where `dropped` and `advisories`
    are lists of (reason, record). An advisory records a failure that policy
    chose not to act on.
    """
    counts = Counter()
    kept, dropped, advisories = [], [], []
    seen_fingerprints = set()
    survivors = []

    for record in records:
        ok, reason = check_schema(record)
        if not ok:
            counts["schema"] += 1
            if policy["schema"] == ENFORCE:
                dropped.append((reason, record))
                continue
            advisories.append((reason, record))

        ok, reason = check_grounded(record)
        if not ok:
            counts["ungrounded"] += 1
            if policy["grounded"] == ENFORCE:
                dropped.append((reason, record))
                continue
            advisories.append((reason, record))

        ratio = compression_ratio(record)
        if not (MIN_COMPRESSION <= ratio <= MAX_COMPRESSION):
            counts["compression"] += 1
            if policy["compression"] == ENFORCE:
                dropped.append((f"compression ratio {ratio:.2f} outside band", record))
                continue
            advisories.append((f"compression ratio {ratio:.2f} outside band", record))

        fingerprint = context_fingerprint(record)
        if fingerprint in seen_fingerprints:
            counts["exact_duplicate"] += 1
            if policy["duplicates"] == ENFORCE:
                dropped.append(("exact duplicate context", record))
                continue
            advisories.append(("exact duplicate context", record))
        seen_fingerprints.add(fingerprint)

        survivors.append(record)

    # Pairwise near-duplicate detection within each domain.
    token_sets = [set(tokens(r["context"])) for r in survivors]
    keep = [True] * len(survivors)
    for i in range(len(survivors)):
        if not keep[i]:
            continue
        for j in range(i + 1, len(survivors)):
            if not keep[j] or survivors[i]["domain"] != survivors[j]["domain"]:
                continue
            if jaccard(token_sets[i], token_sets[j]) >= NEAR_DUP_THRESHOLD:
                keep[j] = False
                counts["near_duplicate"] += 1

    for record, flag in zip(survivors, keep):
        if flag or policy["duplicates"] == ADVISE:
            kept.append(record)
            if not flag:
                advisories.append(("near-duplicate context", record))
        else:
            dropped.append(("near-duplicate context", record))

    return kept, dropped, advisories, counts


print(f"✓ Gate policy for {DATA_SOURCE!r}: {GATE_POLICIES[DATA_SOURCE]}")

✓ Gate policy for 'synthetic': {'schema': 'enforce', 'grounded': 'enforce', 'compression': 'enforce', 'duplicates': 'enforce'}


In [52]:
policy = GATE_POLICIES[DATA_SOURCE]
clean_records, dropped_records, advisory_records, gate_counts = run_quality_gates(raw_records, policy)

print(f"Input records:   {len(raw_records)}")
print(f"Passed gates:    {len(clean_records)}")
print(f"Dropped:         {len(dropped_records)}")
print(f"Advisory flags:  {len(advisory_records)} (failed a check this policy does not enforce)")
print(f"\nAll gate hits:   {dict(gate_counts) or 'none — every record passed every check'}")

if dropped_records:
    print("\nDrop reasons:")
    for reason, count in Counter(reason for reason, _ in dropped_records).most_common(8):
        print(f"  {count:5d}  {reason}")

if advisory_records:
    print("\nAdvisory sample (kept, but worth understanding):")
    for reason, record in advisory_records[:3]:
        print(f"  [{reason}]")
        print(f"      response: {record['response'][:140]}...")

Input records:   1200
Passed gates:    1200
Dropped:         0
Advisory flags:  0 (failed a check this policy does not enforce)

All gate hits:   none — every record passed every check


### Prove the gates actually fire

A validation suite that never rejects anything is indistinguishable from one that does nothing. The synthetic corpus is grounded by construction, so a clean run drops **zero** records — which means the gates are unverified unless you deliberately feed them bad input.

The probe below injects six corrupted records, one per gate, and asserts each is caught. Two deliberate choices:

- It runs under the **strict** policy regardless of `DATA_SOURCE`, because the point is to test the validator functions, not the policy.
- Its six *control* records always come from the generator, even on the Dolly path. The controls have to be known-good under the strict policy, and real Dolly records are not — their human-written summaries paraphrase, so they fail strict groundedness by design. Seeding the probe from the live corpus made this assertion fail on the Dolly path with 8 drops instead of 6: the 6 injected defects plus 2 perfectly good Dolly records. That is the gate-policy asymmetry showing up as a test bug.

Keep this pattern in your own pipelines. It is a regression test for your data-quality code, and it costs almost nothing.

In [53]:
def defective_records(source):
    """Six deliberately broken records, one per gate."""
    hallucinated_entity = dict(source[0])
    hallucinated_entity["response"] = source[0]["response"] + " Analysts at Pinefield Capital rated it a buy."

    hallucinated_number = dict(source[1])
    hallucinated_number["response"] = NUMBER.sub("99999", source[1]["response"], count=1) \
        if NUMBER.search(source[1]["response"]) else source[1]["response"] + " Revenue was 99999 million."

    too_short = dict(source[2])
    too_short["response"] = "It happened."

    empty_target = dict(source[3])
    empty_target["response"] = "   "

    exact_duplicate = dict(source[4])   # identical context to source[4]

    not_compressed = dict(source[5])
    not_compressed["response"] = source[5]["context"]   # copies the input verbatim

    return [hallucinated_entity, hallucinated_number, too_short,
            empty_target, exact_duplicate, not_compressed]


# Controls come from the generator, not from clean_records: they must pass the
# STRICT policy, which real Dolly records do not (see the note above).
# examples_per_domain=1 yields one record per domain, six in total.
probe_controls = synthetic_corpus.generate(examples_per_domain=1, seed=SEED)

strict_policy = GATE_POLICIES["synthetic"]
probe_set = probe_controls + defective_records(probe_controls)
probe_kept, probe_dropped, _, _ = run_quality_gates(probe_set, strict_policy)

print(f"Probe set: {len(probe_set)} records ({len(probe_controls)} good + 6 deliberately broken)")
print(f"Kept: {len(probe_kept)}  Dropped: {len(probe_dropped)}\n")
for reason, _ in probe_dropped:
    print(f"  \u2713 caught -> {reason}")

assert len(probe_kept) == len(probe_controls), (
    f"the {len(probe_controls)} control records should all survive; kept {len(probe_kept)}")
assert len(probe_dropped) == 6, f"expected 6 defects caught, got {len(probe_dropped)}"
print("\n\u2713 All six gates fire on their intended defect, and no control record was lost")

Probe set: 12 records (6 good + 6 deliberately broken)
Kept: 6  Dropped: 6

  ✓ caught -> ungrounded entity: Pinefield
  ✓ caught -> ungrounded number: 99999
  ✓ caught -> response shorter than 8 words
  ✓ caught -> missing or empty field: response
  ✓ caught -> exact duplicate context
  ✓ caught -> compression ratio 1.00 outside band

✓ All six gates fire on their intended defect, and no control record was lost


---
## Section 4: Profile the Dataset

Read the distributions before you spend GPU hours:

- **Length distributions** — context length drives sequence length and therefore training cost. Lab 3B sets `max_input_length=1024` tokens; contexts above that are truncated, which can silently remove the very facts the target summary cites.
- **Compression ratio spread** — a tight band means a consistent task definition. A bimodal spread means you are really training on two different tasks.
- **Domain balance** — an imbalanced corpus produces a model that is good at the majority domain and mediocre elsewhere.
- **Instruction variety** — if one phrasing dominates, the model keys on that exact string instead of the task.

This is where the two corpora look genuinely different, and it is worth running the notebook both ways to see it. Dolly's human-written contexts vary wildly in length and compression; the synthetic corpus is tight and balanced by construction. Neither is automatically better — but the Dolly profile tells you to expect truncation, and the synthetic profile tells you to expect narrow phrasing variety.

In [54]:
profile = pd.DataFrame({
    "domain": [r["domain"] for r in clean_records],
    "instruction": [r["instruction"] for r in clean_records],
    "context_words": [len(tokens(r["context"])) for r in clean_records],
    "response_words": [len(tokens(r["response"])) for r in clean_records],
    "compression": [compression_ratio(r) for r in clean_records],
})

print(f"Length and compression distribution ({DATA_SOURCE})")
display(profile[["context_words", "response_words", "compression"]].describe().round(3))

# Rough token-budget check against Lab 3B's max_input_length of 1024 tokens.
# ~1.3 tokens per word is a reasonable English estimate for a sanity check.
estimated_tokens = profile["context_words"] * 1.3
over_budget = int((estimated_tokens > 1024).sum())
print(f"\nEstimated context tokens: max {estimated_tokens.max():.0f}")
print(f"Records over Lab 3B's 1024-token input budget: {over_budget} "
      f"({over_budget / len(profile):.1%})")
if over_budget:
    print("  These will be truncated during training. If the truncated tail holds facts the")
    print("  target summary cites, those records teach the model to invent them.")

Length and compression distribution (synthetic)


,context_words,response_words,compression
count,1200.000,1200.000,1200.000
mean,89.508,31.168,0.357
std,17.120,4.460,0.068
min,54.000,20.000,0.238
25%,78.000,31.000,0.300
50%,88.000,32.000,0.351
75%,101.000,34.000,0.395
max,136.000,37.000,0.562



Estimated context tokens: max 177
Records over Lab 3B's 1024-token input budget: 0 (0.0%)


In [55]:
print("Records per domain")
display(profile["domain"].value_counts().to_frame("records"))

if profile["domain"].nunique() > 1:
    print("\nCompression ratio by domain")
    display(profile.groupby("domain")["compression"].agg(["mean", "min", "max"]).round(3))

print(f"\nDistinct instruction phrasings: {profile['instruction'].nunique()}")
display(profile["instruction"].value_counts().head(10).to_frame("records"))

Records per domain


,records
domain,
city_infrastructure,200
quarterly_earnings,200
weather_event,200
sports_event,200
research_study,200
product_launch,200



Compression ratio by domain


,mean,min,max
domain,,,
city_infrastructure,0.417,0.327,0.547
product_launch,0.317,0.243,0.420
quarterly_earnings,0.305,0.248,0.381
research_study,0.345,0.270,0.432
sports_event,0.321,0.238,0.436
weather_event,0.434,0.326,0.562



Distinct instruction phrasings: 8


,records
instruction,
Give a concise summary of the passage.,165
Write a short summary of the passage below.,157
Condense the article into one or two sentences.,157
Summarize the following text.,153
Briefly summarize the article.,150
Summarize the key facts in the text below.,142
Read the text and produce a short summary.,142
Provide a brief summary of this report.,134


---
## Section 5: Split and Check for Leakage

Two improvements over a plain random split, both of which matter more for generated data:

**Stratify by domain.** A random split of a multi-domain corpus can under-represent a domain in the test set, and your evaluation then says nothing about it. Dolly has a single stratum, so this degrades gracefully to an ordinary shuffle-and-cut.

**Check for leakage.** This is the characteristic synthetic-data failure. When records are drawn from the same templates and entity pools, two can come out near-identical by chance and land on opposite sides of the split. The model then scores well on test items it effectively memorized. Real scraped data rarely has this problem; a generated corpus has it whenever the pools are too small for the volume requested. Note that leakage **inflates** your evaluation rather than breaking it, so nothing else will tell you it happened.

In [56]:
def stratified_split(records, test_size, seed):
    rng = random.Random(seed)
    by_domain = {}
    for record in records:
        by_domain.setdefault(record["domain"], []).append(record)

    train, test = [], []
    for items in by_domain.values():
        items = list(items)
        rng.shuffle(items)
        cut = int(round(len(items) * (1 - test_size)))
        train += items[:cut]
        test += items[cut:]

    rng.shuffle(train)
    rng.shuffle(test)
    return train, test


train_records, test_records = stratified_split(clean_records, TEST_SIZE, SEED)

print(f"Train: {len(train_records)}  Test: {len(test_records)}")
display(pd.DataFrame({
    "train": Counter(r["domain"] for r in train_records),
    "test": Counter(r["domain"] for r in test_records),
}).fillna(0).astype(int))

Train: 840  Test: 360


,train,test
sports_event,140,60
weather_event,140,60
quarterly_earnings,140,60
city_infrastructure,140,60
product_launch,140,60
research_study,140,60


In [57]:
LEAKAGE_THRESHOLD = 0.90

train_token_sets = [set(tokens(r["context"])) for r in train_records]
leaked = []
for record in test_records:
    test_tokens = set(tokens(record["context"]))
    if any(jaccard(test_tokens, train_tokens) >= LEAKAGE_THRESHOLD
           for train_tokens in train_token_sets):
        leaked.append(record)

print(f"Test records with a >={LEAKAGE_THRESHOLD:.0%} similar context in train: {len(leaked)}")
if leaked:
    print("⚠ Leakage detected — evaluation would be optimistic.")
    if USING_SYNTHETIC:
        print("  Fix at the source: enlarge the entity pools or add sentence phrasings")
        print("  in synthetic_corpus.py, then re-run from Section 2.")
    test_records = [r for r in test_records if r not in leaked]
    print(f"  Removed from the test set. Test size now {len(test_records)}.")
else:
    print("✓ No leakage — evaluation on the test split is meaningful")

Test records with a >=90% similar context in train: 0
✓ No leakage — evaluation on the test split is meaningful


---
## Section 6: Write the Fine-Tuning Package

The `domain` key was useful for stratification and profiling, but it is not part of the training contract — JumpStart instruction tuning expects `instruction`, `context`, and `response` only.

**JSONL is required.** SageMaker fine-tuning reads JSON Lines, one object per line. A CSV or Parquet file with matching column names is not accepted, regardless of how correct its schema looks.

In [58]:
def write_jsonl(records, path):
    """One JSON object per line, training fields only."""
    with open(path, "w", encoding="utf-8") as handle:
        for record in records:
            payload = {field: record[field] for field in TRAINING_FIELDS}
            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")
    return path


train_path = write_jsonl(train_records, "train.jsonl")
evaluation_path = write_jsonl(test_records, "test.jsonl")

for path in (train_path, evaluation_path):
    line_count = sum(1 for _ in open(path, encoding="utf-8"))
    size_kb = os.path.getsize(path) / 1024
    print(f"✓ {path}: {line_count} records, {size_kb:.1f} KB")

print("\nFirst line of train.jsonl:")
with open(train_path, encoding="utf-8") as handle:
    first = json.loads(handle.readline())
print(json.dumps({k: (v[:300] + "..." if len(v) > 300 else v) for k, v in first.items()}, indent=2))

✓ train.jsonl: 840 records, 689.2 KB
✓ test.jsonl: 360 records, 297.3 KB

First line of train.jsonl:
{
  "instruction": "Give a concise summary of the passage.",
  "context": "The Ironworks beat the Kestrels 67 to 61 in a rugby match in Cebu. Concession queues were long at the interval, according to stadium staff. Both sides return to action later in the week. The visiting squad travelled without two of its regular starters. A youth exhibition game preceded the main event...",
  "response": "In Cebu, the Ironworks defeated the Kestrels 67 to 61; Oscar Delgado led with 4 points and attendance was 61200."
}


### Prompt template — registered in MLflow, rendered to S3

A **prompt template** defines how inputs are structured for the model. The same template must be used for training **and** inference: a mismatch is a common and hard-to-diagnose cause of weak fine-tuned results.

That makes the template a first-class input to the model, so it is versioned like one. The template is defined once here, registered in the **MLflow Prompt Registry**, and then *rendered* to the `template.json` the training job needs.

**Why both.** JumpStart instruction fine-tuning reads `template.json` from the training channel directory, on disk, inside the training container — which has no MLflow client and no tracking credentials. So the S3 file is not optional and the registry cannot replace it. What the registry adds is what the S3 object cannot give you:

| | S3 `template.json` | MLflow prompt registry |
|---|---|---|
| Versioning | overwritten in place | numbered versions + commit messages |
| Lineage | none — an untracked file | linked to the run and model that used it |
| Resolution | path | `prompts:/<name>@production` alias |

The direction matters: the registry is the **source of truth**, `template.json` is a build artifact rendered from it. Edit the template here, never the file.

> **Syntax conversion, not a copy.** MLflow prompts use `{{variable}}`; JumpStart's `template.json` is consumed with `str.format()`, which uses `{variable}` and treats a lone brace as a format field. A naive `.replace("{{", "{")` corrupts any literal brace in the prose — `Return JSON like {"k": 1}` becomes a template that raises `KeyError: '"k"'`. `to_format_string()` below lifts the placeholders out, escapes remaining braces, then restores the placeholders, and the cell asserts both renderings agree.

MLflow is optional here. If no tracking URI is configured, the cell reports it and still writes `template.json` — data prep does not hard-fail because an experiment server is down.

In [59]:
import re

# ---------------------------------------------------------------------------
# Source of truth: MLflow {{variable}} syntax.
# ---------------------------------------------------------------------------
# Scoped by USE CASE, not by user or project. The template is a generic
# Alpaca-style instruction scaffold with nothing user- or project-specific in
# it, so one registry entry is shared by every user and every project doing
# this use case. A different use case (classification, extraction) gets its own
# prompt. Matches the use_case Lab 3C records on the model package.
USE_CASE = "text-summarization"
PROMPT_NAME = f"{USE_CASE}-instruct"
PROMPT_ALIAS = "production"

PROMPT_MLFLOW = (
    "Below is an instruction that describes a task, paired with an input that "
    "provides further context. Write a response that appropriately completes "
    "the request.\n\n### Instruction:\n{{instruction}}\n\n### Input:\n{{context}}\n\n"
)
COMPLETION_MLFLOW = " {{response}}"

MUSTACHE = re.compile(r"\{\{\s*(\w+)\s*\}\}")


def to_format_string(text):
    """Convert an MLflow prompt template into a str.format() template.

    Order matters: lift the {{variable}} placeholders out FIRST, then escape
    every remaining literal brace by doubling it, then restore the placeholders
    as single-brace fields. A bare .replace("{{", "{") would turn a literal
    brace in the prose into a format field and corrupt the template.
    """
    found = []

    def _stash(match):
        found.append(match.group(1))
        return f"\x00{len(found) - 1}\x00"

    staged = MUSTACHE.sub(_stash, text)
    staged = staged.replace("{", "{{").replace("}", "}}")
    for index, name in enumerate(found):
        staged = staged.replace(f"\x00{index}\x00", "{" + name + "}")
    return staged, found


# ---------------------------------------------------------------------------
# Register the prompt (best-effort: MLflow is not required for data prep).
# ---------------------------------------------------------------------------
# The tracking URI is read from the .env FILE written by lab0-setup, not from
# os.environ: Studio pre-sets MLFLOW_TRACKING_URI to the domain's DEFAULT app,
# which would quietly register the prompt in the wrong place. Lab 3B resolves
# its app the same way.
prompt_uri = ""
mlflow_tracking_uri = env_from_file("MLFLOW_TRACKING_URI")

if mlflow_tracking_uri:
    try:
        import mlflow
        import mlflow.genai as genai

        mlflow.set_tracking_uri(mlflow_tracking_uri)

        # Idempotent on purpose. A shared prompt has many writers -- every user
        # and project running this lab -- and register_prompt() does not dedupe,
        # so re-running would pile up byte-identical versions and move the alias
        # each time. Register only when the template actually differs.
        current = genai.load_prompt(
            f"prompts:/{PROMPT_NAME}@{PROMPT_ALIAS}",
            allow_missing=True,      # returns None instead of raising
            link_to_model=False,     # no active run here; this is only a lookup
        )
        if current is not None and current.template == PROMPT_MLFLOW:
            prompt_uri = f"prompts:/{PROMPT_NAME}/{current.version}"
            print(f"\u2713 Prompt unchanged \u2014 reusing {prompt_uri}")
            print(f"  alias: prompts:/{PROMPT_NAME}@{PROMPT_ALIAS}")
        else:
            prompt_version = genai.register_prompt(
                name=PROMPT_NAME,
                template=PROMPT_MLFLOW,
                commit_message="Alpaca-style instruction/input/response scaffold "
                               "for instruction fine-tuning",
                tags={
                    # Tags describe the PROMPT, not the run that registered it.
                    # Run-specific facts (which dataset, which lab) belong on the
                    # run -- on a shared entry they would be wrong for everyone
                    # else using it.
                    "completion_format": COMPLETION_MLFLOW,
                    "consumer": "sagemaker-jumpstart-instruction-tuning",
                    "use_case": USE_CASE,
                },
            )
            genai.set_prompt_alias(name=PROMPT_NAME, alias=PROMPT_ALIAS,
                                   version=prompt_version.version)
            prompt_uri = f"prompts:/{PROMPT_NAME}/{prompt_version.version}"
            _what = "Registered" if current is None else "Template changed; registered"
            print(f"\u2713 {_what} {PROMPT_NAME} version {prompt_version.version}")
            print(f"  uri:   {prompt_uri}")
            print(f"  alias: prompts:/{PROMPT_NAME}@{PROMPT_ALIAS}")
    except Exception as exc:
        print(f"⚠ Prompt registration skipped: {type(exc).__name__}: {str(exc)[:160]}")
        print("  template.json is still written below; Lab 3B falls back to the S3 file.")
else:
    print("⚠ No MLFLOW_TRACKING_URI in .env — skipping prompt registration.")
    print("  Run lab0-setup/setup.ipynb to populate it. template.json is still written.")

# ---------------------------------------------------------------------------
# Render template.json for the training channel.
# ---------------------------------------------------------------------------
prompt_format, prompt_vars = to_format_string(PROMPT_MLFLOW)
completion_format, _ = to_format_string(COMPLETION_MLFLOW)

template = {"prompt": prompt_format, "completion": completion_format}
template_path = "template.json"
with open(template_path, "w", encoding="utf-8") as handle:
    json.dump(template, handle)

# The conversion is the one place training and inference formatting can silently
# diverge, so assert the two renderings agree rather than trusting the regex.
_sample = train_records[0]
_via_format = template["prompt"].format(instruction=_sample["instruction"],
                                        context=_sample["context"])
_via_mustache = PROMPT_MLFLOW
for _k in ("instruction", "context"):
    _via_mustache = re.sub(r"\{\{\s*" + _k + r"\s*\}\}", _sample[_k].replace("\\", "\\\\"), _via_mustache)
assert _via_format == _via_mustache, "brace conversion changed the rendered prompt"

print(f"\n✓ template.json rendered from the registry template (variables: {prompt_vars})")
print("  conversion verified: {{var}} and {var} renderings are identical")

rendered = _via_format + template["completion"].format(response=_sample["response"])
print("\nRendered training prompt for the first record:\n")
print(rendered[:900] + ("..." if len(rendered) > 900 else ""))

2026/10/28 18:18:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: text-summarization-instruct, version 1


✓ Registered text-summarization-instruct version 1
  uri:   prompts:/text-summarization-instruct/1
  alias: prompts:/text-summarization-instruct@production

✓ template.json rendered from the registry template (variables: ['instruction', 'context'])
  conversion verified: {{var}} and {var} renderings are identical

Rendered training prompt for the first record:

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Give a concise summary of the passage.

### Input:
The Ironworks beat the Kestrels 67 to 61 in a rugby match in Cebu. Concession queues were long at the interval, according to stadium staff. Both sides return to action later in the week. The visiting squad travelled without two of its regular starters. A youth exhibition game preceded the main event. The officiating crew issued no major penalties in the second half. Oscar Delgado led the winners with 4 poi

---
## Section 7: Upload to S3

Three objects, one location — the exact paths Lab 3B reads for fine-tuning and evaluation.

In [60]:
from sagemaker.core.s3 import S3Uploader

training_input_path = f"{data_location}/{train_path}"
eval_input_path = f"{data_location}/{evaluation_path}"

for path in (train_path, evaluation_path, template_path):
    S3Uploader.upload(path, data_location)

print("✓ Fine-tuning dataset uploaded")
print(f"  Source:           {DATA_SOURCE}")
print(f"  Data location:    {data_location}")
print(f"  Training input:   {training_input_path}")
print(f"  Evaluation input: {eval_input_path}")

✓ Fine-tuning dataset uploaded
  Source:           synthetic
  Data location:    s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset
  Training input:   s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset/train.jsonl
  Evaluation input: s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset/test.jsonl


In [61]:
# Verify the uploaded objects.
from sagemaker.core.s3 import S3Downloader

print("Objects in the fine-tuning dataset location:")
for obj in S3Downloader.list(data_location):
    print(f"  {obj}")

Objects in the fine-tuning dataset location:
  s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset/template.json
  s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset/test.jsonl
  s3://sagemaker-us-east-1-716664005094/Usera/dolly_dataset/train.jsonl


---
## Summary

You published a complete, validated, ready-to-train package to S3:

- `s3://<default-bucket>/<ProfileName>/dolly_dataset/train.jsonl`
- `s3://<default-bucket>/<ProfileName>/dolly_dataset/test.jsonl`
- `s3://<default-bucket>/<ProfileName>/dolly_dataset/template.json`

The path derives from the account/region **default SageMaker bucket** (`sagemaker-<region>-<account-id>`) and your Studio profile name. Lab 3B reads from this same location, so confirm it resolves `bucket` to the same default bucket (`bucket = sess.default_bucket()`).

### What this lab was really about

Acquisition is the part everyone focuses on and the part that matters least. The transferable work is everything after it:

1. **Source selection is a design decision, not a default.** Download when a suitable corpus exists and its license and content survive review. Generate when it does not — which is where most real fine-tuning projects actually start.
2. **Validate regardless of provenance.** A reputable public dataset is not evidence of fitness for your training job. Dolly ships summarization records with empty contexts.
3. **Enforcement depends on who wrote the data.** Strict groundedness is right for a corpus you generate and wrong for human abstractive summaries. A validator encodes an assumption about how the data was produced — make that assumption explicit.
4. **A too-strict validator is also a bug.** The trailing-period and sentence-initial-capital issues in Section 3 would have silently discarded good records and skewed the distribution, with nothing to signal it.
5. **Test your validators.** The injected-defect probe is a regression test for your data quality code. A suite that never rejects anything proves nothing.
6. **Check train/test leakage explicitly.** It inflates your evaluation rather than breaking it, so no other signal will reveal it.
7. **Profile against the training configuration.** Context length only has meaning relative to Lab 3B's `max_input_length`.

### Where the synthetic path runs out

Template-generated text has narrower phrasing variety than human writing. A model fine-tuned on it learns the summarization *behavior* — extract the salient facts, drop the filler, stay grounded — which is enough to show a clear contrast against the base model in Lab 3B. It will not pick up the stylistic range a human corpus provides.

When you need that range and no suitable dataset exists, the next step up is a **teacher model**: prompt a larger LLM to write the contexts and summaries, then run them through these same gates. The gates matter more in that setup, not less — a teacher model will hallucinate, so groundedness becomes a real filter instead of a structural guarantee.

### Next step

Continue to `lab3-model-build/lab-3b-fine-tuning.ipynb` to fine-tune Llama 3.2 3B on this data. It needs no changes: it resolves the same bucket, profile, and prefix this notebook wrote to, whichever source you chose.